# I-E01 — I-N120 + I-S101 Soft-Voting Ensemble

I-N120 nnU-Net과 I-S101 SegResNet의 best epoch 확률맵을 동일 validation 38 case에서 voxel 단위로 평균한다.

- 가중치: `0.5 / 0.5`
- N120 확률맵은 내부 배열 축을 prediction NIfTI로 검증해 복원한다.
- S101의 일부 case는 서로 다른 voxel grid이므로, S101 prediction NIfTI의 affine을 사용해 N120 grid로 선형 resample한다.
- threshold는 lesion F2 우선, `FP/scan ≤ 1.5` 제약으로 선택한다.

> 이 validation 결과는 앙상블 후보 선정용이다. 최종 성능 보고는 threshold를 고정한 뒤 독립 test/hold-out에서 수행한다.


In [1]:
from pathlib import Path
import json, subprocess
import pandas as pd
from IPython.display import display

REPO = Path("/home/stu03/workspace/model")
PYTHON = REPO / ".venv/bin/python"
LABEL_DIR = Path("/home/stu03/workspace/datasets/ischemic/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations")
N120_PROB = REPO / "runs/I-nnunet/Dataset502_ISLES2022/nnUNetTrainerIschemiaMultimodal__IschemiaPlansDualEncoderGatedV1__3d_fullres/fold_0/best_probability_validation"
S101_PROB = REPO / "runs/I-segresnet/I-S101/final_metrics/val_probabilities"
S101_REF = REPO / "runs/I-segresnet/I-S101/final_metrics/val_predictions"
OUT = REPO / "runs/ischemia_ensemble/I-E01_N120_S101"
assert all(x.exists() for x in [PYTHON, LABEL_DIR, N120_PROB, S101_PROB, S101_REF])
print('N120 maps:', len(list(N120_PROB.glob('*.npz'))))
print('S101 maps:', len(list(S101_PROB.glob('*.npz'))))
print('output:', OUT)


N120 maps: 38
S101 maps: 38
output: /home/stu03/workspace/model/runs/ischemia_ensemble/I-E01_N120_S101


## Soft-voting 실행

학습은 수행하지 않는다. 저장된 best probability map만 정렬·평균·평가한다.


In [2]:
command = [
    str(PYTHON), str(REPO / "scripts/ischemia/ensemble/evaluate_soft_voting.py"),
    "--model", f"I-N120={N120_PROB}",
    "--model", f"I-S101={S101_PROB}::{S101_REF}",
    "--label-dir", str(LABEL_DIR),
    "--output-dir", str(OUT),
    "--max-fp-per-scan", "1.5",
]
subprocess.run(command, cwd=REPO, check=True)


{
  "candidate": "soft_vote__I-N120+I-S101",
  "threshold": 0.3,
  "macro_dice_positive": 0.8094789786081708,
  "voxel_recall": 0.8129867421062336,
  "voxel_precision": 0.8493587176903303,
  "lesion_recall": 0.6932270916334662,
  "lesion_precision": 0.8108108108108109,
  "lesion_f1": 0.7474226804123713,
  "lesion_f2": 0.7139340226489415,
  "small_lesion_recall": 0.6398104265402843,
  "fp_per_scan": 1.105263157894737,
  "negative_case_fp_rate": 0.0
}


CompletedProcess(args=['/home/stu03/workspace/model/.venv/bin/python', '/home/stu03/workspace/model/scripts/ischemia/ensemble/evaluate_soft_voting.py', '--model', 'I-N120=/home/stu03/workspace/model/runs/I-nnunet/Dataset502_ISLES2022/nnUNetTrainerIschemiaMultimodal__IschemiaPlansDualEncoderGatedV1__3d_fullres/fold_0/best_probability_validation', '--model', 'I-S101=/home/stu03/workspace/model/runs/I-segresnet/I-S101/final_metrics/val_probabilities::/home/stu03/workspace/model/runs/I-segresnet/I-S101/final_metrics/val_predictions', '--label-dir', '/home/stu03/workspace/datasets/ischemic/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations', '--output-dir', '/home/stu03/workspace/model/runs/ischemia_ensemble/I-E01_N120_S101', '--max-fp-per-scan', '1.5'], returncode=0)

## 결과 확인


In [3]:
summary = json.loads((OUT / "ensemble_summary.json").read_text())
print('Models:', ', '.join(summary['models']))
print('Weights:', summary['weights'])
display(pd.DataFrame([summary['selected']]))
print('Saved ensemble probability maps:', summary['probability_dir'])

diversity = pd.read_csv(OUT / "model_diversity.csv")
display(diversity)


Models: I-N120, I-S101
Weights: equal (1/N)


,candidate,threshold,macro_dice_positive,voxel_recall,voxel_precision,lesion_recall,lesion_precision,lesion_f1,lesion_f2,small_lesion_recall,fp_per_scan,negative_case_fp_rate
0,soft_vote__I-N120+I-S101,0.3,0.809479,0.812987,0.849359,0.693227,0.810811,0.747423,0.713934,0.63981,1.105263,0.0


Saved ensemble probability maps: /home/stu03/workspace/model/runs/ischemia_ensemble/I-E01_N120_S101/probabilities


,model_a,model_b,mean_probability_disagreement
0,I-N120,I-S101,0.000473


In [4]:
sweep = pd.read_csv(OUT / "ensemble_threshold_sweep.csv")
ensemble = sweep[sweep['candidate'].str.startswith('soft_vote__')].copy()
recommended = float(summary['selected']['threshold'])
best_dice = float(ensemble.loc[ensemble['macro_dice_positive'].idxmax(), 'threshold'])
show = sorted({recommended, best_dice, .5})
display(ensemble[ensemble['threshold'].isin(show)][[
    'threshold', 'macro_dice_positive', 'voxel_recall', 'voxel_precision',
    'lesion_recall', 'lesion_precision', 'lesion_f1', 'lesion_f2',
    'small_lesion_recall', 'fp_per_scan', 'negative_case_fp_rate',
]].sort_values('threshold'))


,threshold,macro_dice_positive,voxel_recall,voxel_precision,lesion_recall,lesion_precision,lesion_f1,lesion_f2,small_lesion_recall,fp_per_scan,negative_case_fp_rate
38,0.30,0.809479,0.812987,0.849359,0.693227,0.810811,0.747423,0.713934,0.639810,1.105263,0.0
39,0.35,0.809642,0.805291,0.859190,0.689243,0.826667,0.751726,0.712947,0.635071,1.026316,0.0
42,0.50,0.803726,0.778037,0.883831,0.637450,0.888325,0.742263,0.675610,0.573460,0.578947,0.0


## 산출물

- `ensemble_threshold_sweep.csv`: 단일 모델과 ensemble의 threshold별 지표
- `model_diversity.csv`: 평균 확률 차이
- `ensemble_summary.json`: 선택 threshold 및 지표
- `probabilities/*.npz`: 평균된 2-class probability map
- `probabilities/*.nii.gz`: 추천 threshold로 이진화한 예측 mask
